# レッスン05 - エージェンティックRAG


## セットアップ

このノートブックでは、Microsoft Agent Framework を使用した Agentic RAG (Retrieval-Augmented Generation) パターンを示します。

**前提条件:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — あなたの Azure AI Search サービスエンドポイント
- `AZURE_SEARCH_API_KEY` — あなたの Azure AI Search API キー
- 環境変数を通じて構成された Azure OpenAI のデプロイメント
- Azure CLI での認証済みログイン（`az login`）


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAGとは？

従来のRAGは固定のパイプラインに従います：ドキュメントを検索し、その後応答を生成します。**Agentic RAG**はさらに進んで、エージェントに情報を**いつ**および**どのように**取得するかの自律性を与えます。

Agentic RAGでは、エージェントは以下のことができます：
- 質問に答える前に検索が必要かどうかを**判断する**
- どのデータソースやツールを問い合わせるかを**選択する**
- 取得した結果を**評価し**、最初の検索が不十分な場合は追加検索を実施する
- 複数の検索ステップからの情報を**統合し**、一貫した回答を作成する

これにより、エージェントは固定的な検索してから生成するパイプラインに比べて、より柔軟かつ正確になります。


## 検索ツールの作成

Agentic RAG では、外部データソースがエージェントが必要に応じて呼び出せる**ツール**としてラップされます。これにより、エージェントは検索を必須ステップではなく、実行できるアクションのひとつとして扱うことができます。

以下では、旅行のナレッジベースを定義し、エージェントが目的地情報を調べるために呼び出せるツールとして公開します。


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAGエージェントの構築

ここでは、**必ず回答する前に情報を検索する**ように指示されたエージェントを作成します。エージェントは `search_travel_knowledge` ツールを使用して、トレーニングデータに依存するのではなく、ナレッジベースに基づいて応答を行います。


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 反復的な検索 — メイカー・チェッカーパターン

Agentic RAG の大きな利点は **反復的な検索** です。エージェントは最初の発見を検証、改善、または拡張するために複数回の検索を実行できます。これは「メイカー・チェッカー」ワークフローに似ています。

1. **メイカーステップ**：エージェントは最初の情報を取得し、回答の草案を作成します。
2. **チェッカーステップ**：エージェントは追加の検索を行い、詳細を確認したりギャップを埋めます。

以下では、複数の目的地を比較する必要がある質問がエージェントに与えられ、複数回の検索を促しています。


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## 要約

このレッスンでは、Microsoft Agent Framework を使用して **Agentic RAG** システムを構築する方法を学びました。

- **Agentic RAG** は、エージェントが情報を取得するタイミングを自律的に決定できるようにし、取得を固定的ではなく動的にします。
- **ツールとしてのデータソース**: 外部のナレッジベース（Azure AI Search など）をエージェントが呼び出せるツールとしてラップします。
- **反復的な取得**: メーカー・チェッカーパターンにより、エージェントが複数回の取得ラウンドを実行し、検索、検証、精査を経て最終的な回答を生成します。

実際の運用では、大規模な旅行関連ドキュメントの取得を扱うために、メモリ内の `TRAVEL_KNOWLEDGE_BASE` を実際の Azure AI Search インデックスに置き換えます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類はAI翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を使用して翻訳されました。正確性には努めておりますが、自動翻訳には誤りや不正確な箇所が含まれる可能性があることをご承知おきください。原文はあくまで権威ある情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の使用に起因するいかなる誤解や誤訳についても、責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
